# GNN vs tabular model: where does the graph help, and why?

**Questions**
1. If we reduce the number of classes, does the GNN get better than the tabular model?
2. Where is the GNN better or worse, and what in the graph explains it?

**Method:** no retraining. The saved test probabilities of the two-stage LightGBM (baseline notebook) and of the GNN (GNN notebook) are compared on the same 408,468 test flows. Everything about the graph is computed from the **test** graph (nodes = (IP, port) endpoints, edges = flows), exactly the graph the GNN saw at test time. True labels are used here only to *explain* the results, never as model input.

In [ ]:
import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import f1_score

_fig_count = 0
_plt_show = plt.show


def show_and_save(*args, **kwargs):
    """Show every figure and also keep it as a PNG in the notebook output."""
    global _fig_count
    _fig_count += 1
    plt.savefig(f"/kaggle/working/fig_{_fig_count:02d}.png", dpi=130, bbox_inches="tight")
    _plt_show(*args, **kwargs)


plt.show = show_and_save

DATASET_PATH = "/kaggle/input/datasets/shahiismyname/unsw-nb15-preprocessed-dataset"
find = lambda name: glob.glob(f"/kaggle/input/**/{name}", recursive=True)[0]

test_df = pd.read_parquet(f"{DATASET_PATH}/test.parquet", columns=["label"])
tab = pd.read_parquet(find("two_stage_test_proba.parquet")).loc[test_df.index]    # aligned by row_id
class_names = list(tab.columns)
gnn_proba = np.load(find("gnn_test_proba.npy"))                                   # same row order as test.parquet
graph = pd.read_csv(f"{DATASET_PATH}/test_graph.csv", index_col="row_id").loc[test_df.index]

y = test_df["label"].to_numpy()
pred_tab, pred_gnn = tab.to_numpy().argmax(1), gnn_proba.argmax(1)
ids = list(range(len(class_names)))
print(class_names, "| test flows:", len(y))
print("macro F1  tabular %.4f | GNN %.4f" % (f1_score(y, pred_tab, average="macro"), f1_score(y, pred_gnn, average="macro")))

## 1. Does reducing the classes help the GNN?

Analysis and Backdoor are the classes whose rows mostly have a feature-identical twin with another label. They are merged, folded into Exploits, or removed from scoring, for both models with the same relabelling. If the GNN were held back by those classes, the gap would shrink here.

In [ ]:
A, B, X = (class_names.index(c) for c in ["Analysis", "Backdoor", "Exploits"])
without_ab = [i for i in ids if i not in (A, B)]
keep = ~np.isin(y, [A, B])


def macro(yt, yp, labels):
    return f1_score(yt, yp, labels=labels, average="macro", zero_division=0)


def class_views(p):
    return {
        "10 classes (original)": macro(y, p, ids),
        "Analysis + Backdoor merged (9)": macro(np.where(y == B, A, y), np.where(p == B, A, p), [i for i in ids if i != B]),
        "Analysis + Backdoor folded into Exploits (8)": macro(np.where(np.isin(y, [A, B]), X, y), np.where(np.isin(p, [A, B]), X, p), without_ab),
        "Analysis + Backdoor removed (8)": macro(y[keep], p[keep], without_ab),
    }


views = pd.DataFrame({"tabular": class_views(pred_tab), "GNN": class_views(pred_gnn)})
views["gap (tabular - GNN)"] = views["tabular"] - views["GNN"]
print(views.round(3).to_string())
views[["tabular", "GNN"]].iloc[::-1].plot.barh(figsize=(10, 4.5), color=["darkorange", "steelblue"])
plt.xlabel("test macro F1")
plt.title("Same predictions, different class definitions: the GNN gap does not close")
plt.tight_layout()
plt.show()

## 2. Per-class: where is the GNN better or worse?

In [ ]:
f1 = pd.DataFrame({"tabular": f1_score(y, pred_tab, labels=ids, average=None, zero_division=0),
                   "GNN": f1_score(y, pred_gnn, labels=ids, average=None, zero_division=0)}, index=class_names)
f1["GNN - tabular"] = f1["GNN"] - f1["tabular"]
f1["test rows"] = np.bincount(y, minlength=len(class_names))
print(f1.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={"width_ratios": [3, 2]})
f1[["tabular", "GNN"]].plot.bar(ax=axes[0], color=["darkorange", "steelblue"], width=0.8)
axes[0].set_ylabel("test F1")
axes[0].set_title("Per-class F1")
axes[0].tick_params(axis="x", rotation=45)
f1["GNN - tabular"].sort_values().plot.barh(ax=axes[1], color="gray")
axes[1].axvline(0, color="black", lw=0.8)
axes[1].set_title("GNN minus tabular (right = GNN better)")
plt.tight_layout()
plt.show()

## 3. What does the graph give each flow?

For every test flow we measure two things on the test graph:
- **neighbours** = other flows touching its source or destination endpoint (how rich its neighbourhood is);
- **neighbour purity** = share of those neighbours that carry the flow's own true class (how informative the neighbourhood is). Flows with no neighbour have no purity.

In [ ]:
src_key = graph["srcip"] + ":" + graph["sport"].fillna(-1).astype(int).astype(str)
dst_key = graph["dstip"] + ":" + graph["dsport"].fillna(-1).astype(int).astype(str)
codes, _ = pd.factorize(pd.concat([src_key, dst_key], ignore_index=True))
src_node, dst_node = codes[:len(y)], codes[len(y):]
n_nodes = codes.max() + 1
n_classes = len(class_names)

# flows per (endpoint, class); a flow whose two endpoints coincide is counted once per endpoint
counts = np.zeros((n_nodes, n_classes))
np.add.at(counts, (src_node, y), 1)
np.add.at(counts, (dst_node, y), 1)
degree = counts.sum(1)

neighbours = (degree[src_node] - 1) + (degree[dst_node] - 1)
same_class = (counts[src_node, y] - 1) + (counts[dst_node, y] - 1)
purity = np.where(neighbours > 0, same_class / np.maximum(neighbours, 1), np.nan)

flows = pd.DataFrame({"y": y, "tab_ok": pred_tab == y, "gnn_ok": pred_gnn == y, "neighbours": neighbours, "purity": purity})
flows["neighbour bucket"] = pd.cut(flows["neighbours"], [-1, 0, 9, 99, 999, np.inf], labels=["0", "1-9", "10-99", "100-999", "1000+"])
flows["purity bucket"] = pd.cut(flows["purity"], [-0.01, 0.5, 0.9, 1.0], labels=["< 50% same class", "50-90%", "> 90%"])
print("endpoints:", n_nodes, "| flows with no neighbour: %.1f%%" % (100 * (neighbours == 0).mean()))

## 4. Accuracy by neighbourhood size and by purity

Accuracy per bucket for both models, with the number of flows in each bucket. If the graph helps through neighbours, the GNN should gain on flows with many pure neighbours and lose on flows with none.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, col in zip(axes, ["neighbour bucket", "purity bucket"]):
    tbl = flows.groupby(col, observed=True).agg(flows=("y", "size"), tabular=("tab_ok", "mean"), GNN=("gnn_ok", "mean"))
    print(tbl.round(3).to_string(), "\n")
    tbl[["tabular", "GNN"]].plot.bar(ax=ax, color=["darkorange", "steelblue"], width=0.8)
    ax.set_ylim(0.5, 1.0)
    ax.set_ylabel("accuracy")
    ax.set_xlabel(col)
    ax.set_title(f"Accuracy by {col}")
    ax.tick_params(axis="x", rotation=0)
    for i, n in enumerate(tbl["flows"]):
        ax.text(i, 0.51, f"n={n:,}", ha="center", fontsize=8)
plt.tight_layout()
plt.show()

## 5. Does neighbour purity explain the per-class winners and losers?

One dot per class: x = average neighbour purity of its test flows (do neighbours share its class?), y = GNN F1 minus tabular F1. If neighbours drive the GNN's results, classes with high purity should sit above zero.

In [ ]:
by_class = flows.groupby("y")["purity"].mean()
by_class.index = class_names
plot = pd.DataFrame({"purity": by_class, "diff": f1["GNN - tabular"], "rows": f1["test rows"]})
print(plot.round(3).to_string())
plt.figure(figsize=(8, 5.5))
plt.scatter(plot["purity"], plot["diff"], s=np.sqrt(plot["rows"]) / 2, color="steelblue", alpha=0.7)
for name, row in plot.iterrows():
    plt.annotate(name, (row["purity"], row["diff"]), textcoords="offset points", xytext=(6, 4), fontsize=9)
plt.axhline(0, color="black", lw=0.8)
plt.xlabel("mean neighbour purity (share of neighbouring flows with the same class)")
plt.ylabel("GNN F1 - tabular F1")
plt.title("Per class: do same-class neighbours explain where the GNN wins? (dot size ~ test rows)")
plt.tight_layout()
plt.show()

## 6. Who is right when the two models disagree?

Rows where only one model is correct, split by true class (share of that class's test rows). A large GNN-only share means the neighbourhood supplied something the flow's own features did not.

In [ ]:
only_gnn = pd.Series((flows["gnn_ok"] & ~flows["tab_ok"]).groupby(flows["y"]).mean().to_numpy(), index=class_names)
only_tab = pd.Series((~flows["gnn_ok"] & flows["tab_ok"]).groupby(flows["y"]).mean().to_numpy(), index=class_names)
dis = pd.DataFrame({"only GNN correct": only_gnn, "only tabular correct": only_tab})
print((dis * 100).round(2).to_string())
dis.plot.bar(figsize=(10, 4.5), color=["steelblue", "darkorange"], width=0.8)
plt.ylabel("share of the class's test rows")
plt.title("Rows where exactly one model is right")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()